# Streamlit Dokumentation

## 1. Code-Struktur

Folgende Directories in data-pipeline sind ausschließlich für Streamlit verantwortlich:
 ```
data-pipeline/
├── streamlit_app.py
├── data_utilities/
│   └── database_connector.py
├── dashboard_sections/
│   ├── section_diego.py
│   ├── section_daniel.py
│   ├── section_duong.py
│   ├── section_negar.py
│   └── section_tugba.py
└── taxi_zones/
    ├── taxi_zones.shp
    ├── taxi_zones.shx
    ├── taxi_zones.dbf
    └── ... (weitere Shapefile-Komponenten)
```

1.  **`streamlit_app.py`**: Die Streamlit "Main"-Datei, sie importiert die Dashboardsections (die verschiedenen Userstories) und strukturert sie in Tabs. Die Inhalte werden mit der `render()`-Funktion der Dashboardsections angezeigt.
2.  **`data_utilities/database_connector.py`**: Stellt Funktion bereit die sich mittels SQLAlchemy-Engine mit der Postgres-Datenbank verbindet.
3.  **`dashboard_sections/`**: Jede `.py`-Datei in diesem Ordner repräsentiert eine User Story. Jede Datei enthält eine `render()`-Funktion, die von der `streamlit_app.py` aufgerufen wird.
4.  **`taxi_zones/`**: Enthält die Shape-Datei für die Interaktive Karte (genutzt von `section_diego.py`).

## 2. Datenabruf & Caching

### 2.1 Datenabruf in einer Sektion
Jede Sektion importiert die SQLAlchemy-Engine und nutzt sie in Kombination mit pandas.read_sql_query, um Daten zu laden.

Folgende Schritte finden statt:

1. SQLAlchemy-Engine angeben

2. SQL-Query definieren

3. Query und Engine an pandas.read_sql_query übergeben

*Beispiel (aus section_diego.py):*
```
import pandas as pd
from data_utilities import database_connector

def load_market_share():
    engine = database_connector.get_sqlalchemy_engine()
    query = """
            SELECT p.provider_name, COUNT(*) AS total_trips
            FROM trips t JOIN providers p ON t.provider_id = p.id
            GROUP BY p.provider_name
            """
    df = pd.read_sql_query(query, engine) #
    return df
```

### 2.2 Datenverarbeitung & Caching
Die Performance des Dashboards hängt entscheidend davon ab, wie oft und wie viele Daten aus der Datenbank geladen werden. Bei den knapp 20Mio Einträgen im Quell-Dataset ist Caching unerlässlich. Dazu verwenden wir den Streamlit-Decorator @st.cache_data. Dieser "merkt" sich das und die Daten müssen nicht ständig neu geladen werden.

```
@st.cache_data(show_spinner="Lade Preisvergleichsdaten...", ttl=600)
def load_price_distribution_data(distance_range: tuple):
    ...
```

**Parameter:**
`show_spinner`: Zeigt eine Ladekringel in der UI an während die Funktion und die SQL-Abfrage läuft.
`ttl`: (Time-To-Live) Weist Streamlit an, den Cache-Eintrag nach 600 Sekunden (10 Minuten) zu invalidieren. Das Dashboard wird so zur Sicherheit nach den 10 Minuten aktualisiert.

Parameter-Tracking: Der Cache ist an die Argumente der Funktion gebunden. load_price_distribution_data((3.0, 10.0)) wird nur einmal ausgeführt. Ändert der User den Slider auf (4.0, 11.0), wird die Funktion einmal neu ausgeführt und das Ergebnis für diese neuen Parameter gespeichert.

#### 2.3 Aggregation und Sampling

Aggregation in SQL (Bevorzugt): Für die meisten Diagramme (Heatmap, Barchart, KPIs) wird die Aggregation (AVG, GROUP BY, COUNT) direkt in der PostgreSQL-Datenbank durchgeführt. So werden nur wenige hundert Zeilen an Python/Pandas gesendet.

Sampling in SQL (Für Rohdaten-Plots): Für Visualisierungen, die Rohdaten benötigen (Boxplot, Scatter-Plot), macht eine Aggregation keinen Sinn. Stattdessen werden Samples gezogen, die die knapp 20mio einträge repräsentieren sollen (z.B in section_diego.py Zeile 20)

#### 2.4 Transformation in Pandas
Nach dem Laden werden die Daten in Pandas weiterverarbeitet.
Die verschiedenen Plots werden dann in Streamlit auf Basis der resultierenden Pandas-Dataframes dargestellt.

#### 2.5 Kartendaten
Die Kartendaten liegen als Shapefile vor und können mithilfe von Geopandas eingelesen werden. Dieses speichert die Geometrien als Koordinatenfolgen ab, genau wie Pandas auch als Dataframe bzw. Geodataframe.

# 3. Dashboardstruktur

### 3.1 streamlit_app.py
Diese Datei dient nur als Container. Sie nutzt st.tabs, um die Sektionen der einzelnen Userstories voneinander zu trennen. Jede Sektion wird in einen eigenen Tab geladen und ihre render()-Funktion aufgerufen.


### 3.2 Interne Sektions-Struktur

Das am häufigsten verwendeten Muster zur Anordnung sind st.tabs und st.columns. Damit lassen sich Inhalte in derselben Zeile in verschiedenen Spalten anzeigen.
Beispielsweise KPIs (st.metric) oder natürlich auch Plots. Tabs und Spalten werden wie folgt definiert:

```
tab_overview, tab_detail, tab_docs = st.tabs([
    "Übersicht",
    "Anbieter-Detail",
    "Dokumentation"
])

with tab_overview:
    # Code für die Übersichts-KPIs...
with tab_detail:
    # Code für die Detail-Analyse...
```
```
row2_col1, row2_col2 = st.columns(2)
row3_col1, row3_col2 = st.columns(2)

with row2_col1:
    # Code für den Boxplot...
with row2_col2:
    # Code für den Scatter-Plot...
```

In den Tabs und Sections können dann die Inhalte, also KPIs, Plots, Interaktive Slider, Radiobuttons etc. eingefügt werden.